# Stage II (vessels) - 08. Vessel masks for the Stage I-routed images

In the cascade, Stage II runs only on images that pass the Stage I gate. This notebook writes one
binary vessel mask per routed image, in the form that LG-DRG (Stage III) reads. Steps:

1. **Configuration** - input folder (images routed by Stage I) and output folder.
2. **Deployed model** - `best_dice.pth` (EMA weights) and threshold 0.46.
3. **Mask generation** - for every image: FOV mask, CLAHE + green enhancement, square FOV crop,
   512 x 512, flip TTA, FOV masking, mapping back to the original resolution, binarisation,
   radius-1 opening and removal of components below 30 pixels.
4. **Check** - every routed image has a mask that LG-DRG can resolve.

**Naming expected by LG-DRG.** LG-DRG looks up the vessel mask of image `<stem>.<ext>` as
`<mask_dir_vessel>/<stem>_mask.<ext>` (`mask_suffix["vessel"] = "_mask"`, extensions tried in
order `.png, .jpg, .ppm, .jpeg, .tif, .tiff`, then a glob). Masks are therefore written as
`<stem>_mask.png` (uint8, 0/255, original image size) into a folder used only for vessels
(the MA masks use the same suffix, so the two must live in separate folders).

## Step 1. Configuration

In [ ]:
from pathlib import Path

# Images routed to Stage II by the Stage I gate (folder "dr_from_test" written by
# hierarchiretina.stage1.gate.route_test_pool).
INPUT_DIR = Path("outputs/stage1_gate/dr_from_test")
# Vessel masks for LG-DRG, written next to the lesion masks (<root>/vessel/<stem>_mask.png).
# For the LG-DRG training masks, run again with INPUT_DIR = data/stage3/dev_images and
# OUTPUT_DIR = data/stage3/dev_masks/vessel.
OUTPUT_DIR = Path("outputs/stage2_masks/test/vessel")
BEST_CKPT = Path("checkpoints/stage2/swinhrunetpp/best_dice.pth")

USE_TTA = True

In [ ]:
import pandas as pd
import torch
from tqdm.auto import tqdm

from hierarchiretina.stage2.engine_vessel import load_best_model_and_threshold
from hierarchiretina.stage2.swinhrunetpp import IMG_SIZE, build_swinhrunetpp
from hierarchiretina.stage2.vessel_data import VALID_EXT
from hierarchiretina.stage2.vessel_inference import VESSEL_MASK_SUFFIX, generate_vessel_masks

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Step 2. Deployed model and threshold

In [ ]:
model = build_swinhrunetpp(pretrained=False, img_size=IMG_SIZE)
model, THR, info = load_best_model_and_threshold(BEST_CKPT, model)
model = model.to(DEVICE).eval()
info

## Step 3. Mask generation

In [ ]:
result = generate_vessel_masks(model, INPUT_DIR, OUTPUT_DIR, THR, DEVICE, img_size=IMG_SIZE,
                               use_tta=USE_TTA, suffix=VESSEL_MASK_SUFFIX, progress=tqdm)
print(f"{result['saved']} of {result['n_images']} masks written to {OUTPUT_DIR}; "
      f"{len(result['failed'])} failed")
result["failed"][:10]

## Step 4. Check that every routed image has a mask under the LG-DRG naming

In [ ]:
stems = sorted(p.stem for p in INPUT_DIR.iterdir() if p.suffix.lower() in VALID_EXT)
missing = [s for s in stems if not (OUTPUT_DIR / f"{s}{VESSEL_MASK_SUFFIX}.png").exists()]
pd.Series({"routed images": len(stems), "masks found": len(stems) - len(missing),
           "missing": len(missing)})